# LongFlow — score the clean-frames ablation on GPU

Runtime: **any GPU**. Reads `cleanabl_eval.zip` from Drive root. Prints the
pre-registered verdict (NOTES "CLEAN-FRAMES ABLATION PRE-REGISTRATION"):
did training on clean-only frames recover the closed-loop identity the
mixed pool lost?


In [ ]:
# ===== COLD START — run me first, wait for READY =====
NOTEBOOK_VERSION = "Score clean ablation (GPU) v1.0 (2026-08-18)"
print(f"*** {NOTEBOOK_VERSION} ***")
!pip install -q faster-whisper jiwer whisper-normalizer speechbrain praat-parselmouth

import torch
assert torch.cuda.is_available(), "no GPU — pick a GPU runtime"
import glob, json, os, sys, zipfile
import numpy as np
import soundfile as sf

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1
from src.eval.metrics import clip_metrics, _ecapa, _normalizer, _whisper

from google.colab import drive
drive.mount("/content/drive")
candidates = glob.glob("/content/drive/MyDrive/cleanabl_eval*.zip")  # root only, NOT recursive
assert candidates, "no cleanabl_eval*.zip in Drive root"
zip_path = candidates[0]
print(f"using {zip_path} ({os.path.getsize(zip_path)/1e9:.2f} GB)")
AUD = "/content/eval_audio"
os.makedirs(AUD, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(AUD)
print(f"extracted {len(os.listdir(AUD))} entries")
print("READY")


In [ ]:
# ===== Held-out (20000:C) + closed loop vs pre-registered bars =====
import jiwer
import torchaudio.functional as taf

with open(f"{AUD}/manifest.json") as f:
    manifest = json.load(f)
results = {"held_out": {}, "closed_loop": {}, "verdicts": {}}

entries = manifest["checkpoints"]["20000:C"]
rows = []
for e in entries:
    m = clip_metrics(f"{AUD}/{e['audio']}", e["text"],
                     f"{AUD}/{e['teacher_audio']}", device="cuda")
    rows.append({**m, "utt_id": e["utt_id"], "target_words": e["target_words"]})
wers = sorted(r["wer"] for r in rows)
sims = sorted(r["speaker_sim"] for r in rows)
by_bin = {}
for r in rows:
    by_bin.setdefault(r["target_words"], []).append(r["wer"])
results["held_out"]["20000:C"] = {
    "n": len(rows), "wer_median": wers[len(wers) // 2],
    "sim_median": sims[len(sims) // 2],
    "wer_by_bin": {b: sorted(v)[len(v) // 2] for b, v in sorted(by_bin.items())},
    "rows": rows,
}
o = results["held_out"]["20000:C"]
print(f"20000:C (clean pool): n={o['n']}  wer_med={o['wer_median']:.3f}  "
      f"sim_med={o['sim_median']:.3f}  by_bin={ {b: round(v,3) for b,v in o['wer_by_bin'].items()} }")
print("reference 20000:B (mixed pool): wer 0.163 / sim 0.901 — a small dip is expected (56% of frames)")

with open(f"{AUD}/cleanabl_report.json") as f:
    report = json.load(f)
WIN, HOP = 4.0, 2.0
ecapa = _ecapa("cuda")
n = _normalizer()

script_words = []
for line in report["cl_script"].splitlines():
    if ":" in line:
        line = line.split(":", 1)[1]
    script_words.append(line.strip())
SCRIPT = n(" ".join(w for w in script_words if w))
N_SCRIPT = len(SCRIPT.split())

def win_embs(path):
    x, sr = sf.read(path, dtype="float32")
    x16 = taf.resample(torch.from_numpy(x), sr, 16000).numpy()
    dur = len(x) / sr
    E, ts = [], []
    for i in range(int((dur - WIN) // HOP) + 1):
        seg = x16[int(i * HOP * 16000): int((i * HOP + WIN) * 16000)]
        if len(seg) < 16000:
            break
        emb = ecapa.encode_batch(torch.from_numpy(seg)[None].to("cuda"))[0, 0]
        E.append(emb.detach().cpu())
        ts.append(i * HOP)
    return torch.stack(E), np.array(ts), dur

def transcribe(path):
    segs, _ = _whisper("cuda").transcribe(str(path), language="en", beam_size=1)
    return " ".join((s.text or "").strip() for s in segs)

ref_E, _, _ = win_embs(f"{AUD}/t1_turnsplit_p0.wav")
ref = ref_E.median(0).values

def score(path):
    E, ts, dur = win_embs(path)
    sim = torch.nn.functional.cosine_similarity(E, ref[None], dim=-1).numpy()
    voice = sim >= 0.5
    horizon = 0.0
    for i in range(len(ts)):
        if voice[i]:
            horizon = ts[i] + WIN
    hyp = n(transcribe(path))
    nw = len(hyp.split())
    return {"duration_s": round(dur, 1),
            "wer_vs_script": round(jiwer.wer(SCRIPT, hyp) if hyp else 1.0, 3),
            "coverage_pct": round(100 * min(nw, N_SCRIPT) / N_SCRIPT, 1),
            "voice_pct": round(100 * float(voice.mean()), 1),
            "horizon_s": float(horizon),
            "sim_median": round(float(np.median(sim)), 3),
            "sim_final_third": round(float(np.median(sim[-max(1, len(sim) // 3):])), 3),
            "rate_wpm": round(60 * nw / dur, 1)}

for p in sorted(glob.glob(f"{AUD}/closed_loop/*.wav")):
    tag = os.path.basename(p)[:-4]
    results["closed_loop"][tag] = score(p)
    print(f"{tag}: {results['closed_loop'][tag]}", flush=True)

# ---- pre-registered verdict ----
ARM_B = {"wer": (0.116, 0.119), "sim": (0.207, 0.297)}  # HV2 mixed-pool rows
GN8 = {"wer": 0.031, "sim": 0.522, "voice": 61.7}
c = results["closed_loop"]
s = [c.get("cleanabl_cfg_heun8_s0"), c.get("cleanabl_cfg_heun8_s1")]
if all(s):
    if all(r["wer_vs_script"] <= 0.07 and r["sim_median"] >= 0.42 for r in s):
        v = ("NOISED FRAMES CULPRIT — clean-trained base back in the GN8 band; "
             "offline base trains CLEAN from now on; noise-robustness belongs to stage-2 on-policy data")
    elif all(
        r["sim_median"] >= max(ARM_B["sim"]) + 0.10 or r["wer_vs_script"] <= min(ARM_B["wer"]) - 0.05
        for r in s
    ):
        v = ("PARTIAL — noised frames contribute but are not the whole regression; "
             "stage-2 proceeds from the clean-trained base; residual is register/prompt-mix")
    else:
        v = ("NO RECOVERY — noised frames acquitted; suspect the long-form register / "
             "multi-prompt mix; investigate BEFORE stage-2")
    results["verdicts"]["clean_ablation"] = v
    print("\nVERDICT:", v)

with open("/content/drive/MyDrive/cleanabl_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print("metrics on Drive root: cleanabl_metrics.json")
print("\nListening (never skipped): both closed-loop renders — is the 'underwater' gone?")
